<a href="https://colab.research.google.com/github/Gidayi-dev/Dummy/blob/main/vaccination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score

In [33]:
# from google.colab import files
# upoloaded = files.upload()

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

train = pd.read_csv('/content/Train.csv')
test = pd.read_csv('/content/Test.csv')

print(f"Train shape : {train.shape}")
print("Label Distribution:\n", train['label'].value_counts())

TEXT_COL = 'safe_text'

Train shape : (10001, 4)
Label Distribution:
 label
 0.000000    4908
 1.000000    4053
-1.000000    1038
 0.666667       1
Name: count, dtype: int64


In [34]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
  if pd.isna(text):
    return ""
  text = text.lower()
  text = re.sub(r'<user>|<url>', '', text)
  text = re.sub(r'&\w+;', '', text)
  text = re.sub(r'#(\w+)', r'\1', text)
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'\d+', '', text)
  text = re.sub(r'\s+', ' ', text).strip()
  tokens = text.split()
  tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
  return ' '.join(tokens)

In [35]:
train = train[train['label'].isin([-1.0, 0.0, 1.0])].dropna(subset=['label'])
train['clean_text'] = train[TEXT_COL].apply(clean_tweet)
test['clean_text'] = test[TEXT_COL].apply(clean_tweet)

X = train['clean_text']
y = train['label'].astype(int)

tfidf =TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(X)
X_test_tfidf = tfidf.transform(test['clean_text'])

print(f"\nTF-IDF matrix: {X_tfidf.shape[0]} tweets x {X_tfidf.shape[1]} features")



TF-IDF matrix: 9999 tweets x 10000 features


In [36]:
X_tr, X_val, y_tr, y_val = train_test_split(X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=1.0),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
}

print("\n=== MODEL COMPARISON ===")
results = []
for name, model in models.items():
  model.fit(X_tr, y_tr)
  preds = model.predict(X_val)
  acc = (preds == y_val).mean()
  f1 = f1_score(y_val, preds, average='weighted')
  results.append({'Model': name, 'Accuracy': round(acc,4), 'F1 (weighted)': round(f1, 4)})
  print(f"\n{name}: Accuracy={acc:.4f}, F1={f1:.4f}")
  print(classification_report(y_val, preds, target_names=['Anti(-1)','Neutral(0)','Pro(1)'], zero_division=0))

print(pd.DataFrame(results).sort_values(by='F1 (weighted)', ascending=False).to_string(index=False))


=== MODEL COMPARISON ===

Logistic Regression: Accuracy=0.7370, F1=0.7190
              precision    recall  f1-score   support

    Anti(-1)       0.69      0.18      0.28       207
  Neutral(0)       0.78      0.82      0.80       982
      Pro(1)       0.69      0.78      0.73       811

    accuracy                           0.74      2000
   macro avg       0.72      0.59      0.61      2000
weighted avg       0.73      0.74      0.72      2000


Naive Bayes: Accuracy=0.7070, F1=0.6757
              precision    recall  f1-score   support

    Anti(-1)       0.80      0.04      0.07       207
  Neutral(0)       0.77      0.77      0.77       982
      Pro(1)       0.64      0.80      0.71       811

    accuracy                           0.71      2000
   macro avg       0.74      0.54      0.52      2000
weighted avg       0.72      0.71      0.68      2000


Random Forest: Accuracy=0.7290, F1=0.7116
              precision    recall  f1-score   support

    Anti(-1)       0.65 

In [37]:
best_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
best_model.fit(X_tfidf, y)
preds = best_model.predict(X_test_tfidf)

print("\nTest prediction distribution:")
for val, name in [(-1, 'Anti-vax'), (0, 'Neutral'), (1, 'Pro-vax')]:
  count = (preds == val).sum()
  print(f"{name}: {count} ({count/len(preds)*100:.1f}%)")


Test prediction distribution:
Anti-vax: 172 (3.3%)
Neutral: 2557 (49.4%)
Pro-vax: 2448 (47.3%)


In [38]:
submission = pd.DataFrame({'tweet_id': test['tweet_id'], 'label': preds})
submission.to_csv('vaccination_submission.csv', index=False)
print(f"\nSaved vaccination_submission.csv - {len(submission)} rows")
print(submission.head())


Saved vaccination_submission.csv - 5177 rows
   tweet_id  label
0  00BHHHP1      1
1  00UNMD0E      1
2  01AXPTJF      0
3  01HOEQJW      1
4  01JUKMAO      0


In [39]:
from google.colab import files
files.download('vaccination_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>